# EDA — Amazon Products Dataset 2023

**Project:** A Controlled Architectural Ablation of Reactive and Planning-Based LLM Agents on Constrained Multi-Step Web Shopping Tasks

**Purpose:** Understand the raw Amazon Products Dataset 2023 (1.4M products), surface data-quality issues, and produce evidence-based decisions for the pre-processing pipeline that will generate the curated ~500-product catalogue.

**Scope:** This notebook covers EDA only. The pre-processing pipeline (implemented as Python modules) and the curated catalogue artefact are produced separately. The task suite is generated in a follow-up step against the cleaned catalogue.

**Target categories (locked):** Headphones & Earbuds, Camera & Photo, Cell Phones & Accessories, Watches (Men's + Women's combined), Laptop Accessories.

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Display config
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 80)
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.dpi'] = 100

# Paths
DATA_RAW = Path('../data/raw')
PRODUCTS_CSV = DATA_RAW / 'amazon_products.csv'
CATEGORIES_CSV = DATA_RAW / 'amazon_categories.csv'

# Target categories (locked after initial category exploration)
TARGET_CATEGORIES = {
    71:  'Headphones & Earbuds',
    79:  'Camera & Photo',
    75:  'Cell Phones & Accessories',
    113: 'Watches (Men\'s)',     # Will be merged with 121 into one 'Watches' bucket
    121: 'Watches (Women\'s)',
    65:  'Laptop Accessories',
}

# Bucket mapping: raw category_id -> our 5-category label used downstream
BUCKET_MAP = {
    71:  'Headphones',
    79:  'Cameras',
    75:  'Phones',
    113: 'Watches',
    121: 'Watches',
    65:  'LaptopAccessories',
}

print(f'Products CSV exists: {PRODUCTS_CSV.exists()}')
print(f'Categories CSV exists: {CATEGORIES_CSV.exists()}')

## 1. Load & Inspect Raw Data

First — load the full products CSV and the categories lookup. Then look at shape, dtypes, and a sample.

In [ ]:
# Load full products dataset (this may take 10-20 seconds for ~1.4M rows)
df = pd.read_csv(PRODUCTS_CSV)
cats = pd.read_csv(CATEGORIES_CSV)

print(f'Products: {len(df):,} rows, {df.shape[1]} columns')
print(f'Categories: {len(cats):,} rows')
print()
print('Memory usage:', df.memory_usage(deep=True).sum() / 1024**2, 'MB')

In [ ]:
# Column dtypes and non-null counts
df.info()

In [ ]:
# Sample rows
df.head(5)

In [ ]:
# Numeric column summary across the whole dataset (for context, before filtering)
df[['stars', 'reviews', 'price', 'listPrice', 'boughtInLastMonth']].describe()

**Section 1 observations to note:**
- Total row count (expect ~1.4M)
- Any unexpected dtypes (e.g. price as object instead of float)
- Initial impression of price/rating ranges

## 2. Category Filtering

Filter the 1.4M products down to just our six target category IDs (which will collapse into 5 final buckets after merging the two watch categories).

In [ ]:
# Filter to target categories only
df_targets = df[df['category_id'].isin(TARGET_CATEGORIES.keys())].copy()
df_targets['bucket'] = df_targets['category_id'].map(BUCKET_MAP)

print(f'After category filter: {len(df_targets):,} rows '
      f'({len(df_targets)/len(df)*100:.2f}% of original)')
print()
print('Counts per raw category:')
for cid, name in TARGET_CATEGORIES.items():
    n = (df_targets['category_id'] == cid).sum()
    print(f'  {cid:>4} {name:<32} {n:>7,}')
print()
print('Counts per final bucket (5 categories):')
print(df_targets['bucket'].value_counts().to_string())

In [ ]:
# Visualize bucket distribution
fig, ax = plt.subplots(figsize=(8, 4))
df_targets['bucket'].value_counts().plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Number of products')
ax.set_title('Raw product counts per target bucket (before cleaning)')
plt.tight_layout()
plt.show()

## 3. Missing Values

For each essential attribute, how many records have missing values within our target categories? This determines how many rows we'd lose if we drop incomplete records (which we will, per the pre-processing plan).

In [ ]:
# Missing value counts per column, overall and per bucket
essential_cols = ['title', 'price', 'stars', 'reviews', 'category_id']

print('Overall missing values within target categories:')
missing = df_targets[essential_cols].isna().sum()
missing_pct = (df_targets[essential_cols].isna().mean() * 100).round(2)
summary = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
print(summary)
print()

# Note: in this dataset, 'missing' for stars/reviews/price often appears as 0 
# rather than NaN. We check that next.
print('Zero-value counts (often a proxy for missing):')
for col in ['price', 'stars', 'reviews']:
    zero_count = (df_targets[col] == 0).sum()
    zero_pct = (df_targets[col] == 0).mean() * 100
    print(f'  {col:<10} zeros: {zero_count:>7,} ({zero_pct:.2f}%)')

In [ ]:
# Per-bucket missing/zero analysis
print('Missing or zero values per bucket:')
print()
rows = []
for bucket in df_targets['bucket'].unique():
    sub = df_targets[df_targets['bucket'] == bucket]
    rows.append({
        'bucket': bucket,
        'total': len(sub),
        'price_zero': (sub['price'] == 0).sum(),
        'stars_zero': (sub['stars'] == 0).sum(),
        'reviews_zero': (sub['reviews'] == 0).sum(),
        'title_na': sub['title'].isna().sum(),
    })
bucket_missing = pd.DataFrame(rows).set_index('bucket')
bucket_missing['after_clean_estimate'] = bucket_missing.apply(
    lambda r: r['total'] - max(r['price_zero'], r['stars_zero'], r['reviews_zero']),
    axis=1
)
print(bucket_missing.to_string())

**Decision point:** Records where `price == 0`, `stars == 0`, or `reviews == 0` are effectively missing — a real product has all three. The pre-processing pipeline will drop these (no imputation, since we don't want synthetic values in a real-data study). The `after_clean_estimate` column above predicts how many products remain per bucket after this cleaning step.

## 4. Duplicates

Are there duplicate ASINs (Amazon Standard Identification Numbers) or duplicate titles? Both can occur in scraped data: ASIN duplicates from scraping errors, title duplicates from sellers listing identical products under different ASINs.

In [ ]:
# ASIN duplicates within targets
asin_dups = df_targets.duplicated(subset='asin').sum()
title_dups = df_targets.duplicated(subset='title').sum()

print(f'Duplicate ASINs (within target categories): {asin_dups:,}')
print(f'Duplicate titles (within target categories): {title_dups:,}')
print()
print(f'Unique ASINs: {df_targets["asin"].nunique():,} / {len(df_targets):,} total')
print(f'Unique titles: {df_targets["title"].nunique():,} / {len(df_targets):,} total')

In [ ]:
# Show a sample of duplicate titles, if any — what do they look like?
if title_dups > 0:
    dup_titles = df_targets[df_targets.duplicated(subset='title', keep=False)]
    sample_title = dup_titles['title'].value_counts().index[0]
    print(f'Example of a duplicated title (appears {(dup_titles["title"] == sample_title).sum()} times):')
    print(f'  "{sample_title}"')
    print()
    print('Sample rows for that title:')
    print(dup_titles[dup_titles['title'] == sample_title].head(3)[['asin', 'title', 'price', 'stars', 'reviews']].to_string())
else:
    print('No duplicate titles found.')

## 5. Price Distribution

What does the price distribution look like per bucket? This determines what constraint thresholds will be meaningful for tasks (e.g. "under €X" needs to actually filter out some products).

Note: the dataset's `price` is in USD. We'll treat amounts as numeric currency-agnostic values for EDA; the report acknowledges this.

In [ ]:
# Filter to non-zero prices for distribution analysis
df_pricing = df_targets[df_targets['price'] > 0].copy()

print('Price summary per bucket (non-zero prices only):')
print(df_pricing.groupby('bucket')['price'].describe().round(2).to_string())

In [ ]:
# Price distribution per bucket — clip extreme outliers for visualisation
fig, axes = plt.subplots(1, 5, figsize=(20, 4), sharey=False)
buckets = sorted(df_pricing['bucket'].unique())
for ax, bucket in zip(axes, buckets):
    sub = df_pricing[df_pricing['bucket'] == bucket]
    # Clip top 1% to prevent extreme outliers dominating the plot
    p99 = sub['price'].quantile(0.99)
    sub_clipped = sub[sub['price'] <= p99]
    ax.hist(sub_clipped['price'], bins=40, color='steelblue', edgecolor='white')
    ax.set_title(f'{bucket}\n(n={len(sub):,}, p99={p99:.0f})')
    ax.set_xlabel('Price')
    ax.set_ylabel('Count')
plt.suptitle('Price distribution per bucket (clipped at 99th percentile)', y=1.02)
plt.tight_layout()
plt.show()

## 6. Rating & Review Distribution

Constraint thresholds for `stars` and `reviews` need to be chosen against the actual distributions. If 95% of products have 4+ stars, then "4+ stars" is not a meaningful constraint.

In [ ]:
# Stars distribution (non-zero only)
df_rated = df_targets[df_targets['stars'] > 0].copy()

print('Stars summary per bucket (rated products only):')
print(df_rated.groupby('bucket')['stars'].describe().round(2).to_string())
print()
print('Fraction of products with stars >= 4.0 per bucket:')
for bucket in sorted(df_rated['bucket'].unique()):
    sub = df_rated[df_rated['bucket'] == bucket]
    frac = (sub['stars'] >= 4.0).mean()
    print(f'  {bucket:<20} {frac:.1%}')

In [ ]:
# Reviews distribution — heavily right-skewed, so check on log scale
df_reviewed = df_targets[df_targets['reviews'] > 0].copy()

print('Reviews summary per bucket (reviewed products only):')
print(df_reviewed.groupby('bucket')['reviews'].describe(percentiles=[.25, .5, .75, .9, .99]).round(1).to_string())
print()
print('Fraction of products with reviews >= 1000 per bucket:')
for bucket in sorted(df_reviewed['bucket'].unique()):
    sub = df_reviewed[df_reviewed['bucket'] == bucket]
    frac = (sub['reviews'] >= 1000).mean()
    print(f'  {bucket:<20} {frac:.1%}')

In [ ]:
# Combined stars vs reviews scatter per bucket
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, bucket in zip(axes, sorted(df_targets['bucket'].unique())):
    sub = df_targets[(df_targets['bucket'] == bucket) & 
                     (df_targets['stars'] > 0) & 
                     (df_targets['reviews'] > 0)]
    # Sample down for visualisation if too many points
    if len(sub) > 5000:
        sub = sub.sample(5000, random_state=42)
    ax.scatter(sub['reviews'], sub['stars'], alpha=0.3, s=8)
    ax.set_xscale('log')
    ax.set_title(bucket)
    ax.set_xlabel('Reviews (log scale)')
    ax.set_ylabel('Stars')
    ax.set_ylim(0, 5.2)
plt.suptitle('Stars vs Reviews per bucket', y=1.02)
plt.tight_layout()
plt.show()

## 7. Cell Phones Deep-Dive

Category 75 ("Cell Phones & Accessories") mixes actual phones with cases, chargers, screen protectors, etc. We need to know whether this is a problem for our tasks. Let's look at common terms in the titles.

If the bulk of category 75 is accessories (which is likely, given the title suggests "& Accessories"), we have two options:
1. **Treat the whole bucket as "phones & accessories"** — adjust task constraints to match the price range of the bucket (e.g. tasks reference "phone accessory under €30" instead of "phone under €900")
2. **Heuristically filter to actual phones** — exclude titles containing keywords like "case", "charger", "screen protector"

Let's see the data first.

In [ ]:
phones = df_targets[df_targets['bucket'] == 'Phones'].copy()
print(f'Total products in Phones bucket: {len(phones):,}')
print()
print('Sample of titles:')
for title in phones['title'].sample(15, random_state=42).values:
    print(f'  - {title[:120]}')

In [ ]:
# Heuristic accessory detection
accessory_keywords = ['case', 'cover', 'charger', 'screen protector', 'cable', 
                      'adapter', 'holder', 'mount', 'stand', 'sticker', 'skin',
                      'lanyard', 'strap', 'wallet', 'sleeve', 'pouch']

# Build regex pattern for case-insensitive match
pattern = '|'.join(accessory_keywords)
phones['is_accessory'] = phones['title'].str.lower().str.contains(pattern, na=False)

accessory_pct = phones['is_accessory'].mean() * 100
print(f'Estimated accessories in Phones bucket: {phones["is_accessory"].sum():,} '
      f'({accessory_pct:.1f}%)')
print()
print('Likely actual phones (sample):')
actual_phones = phones[~phones['is_accessory']]
for title in actual_phones['title'].sample(min(10, len(actual_phones)), random_state=42).values:
    print(f'  - {title[:120]}')
print()
print(f'Likely actual phones in bucket: {len(actual_phones):,}')

In [ ]:
# Price ranges for accessories vs likely phones (sanity check)
print('Price distribution — accessories vs likely phones:')
print(phones[phones['price'] > 0].groupby('is_accessory')['price'].describe().round(2).to_string())

**Decision point:** Based on the count and price distribution above, we'll decide in the pre-processing step whether to:
- **Filter to likely-phones only** (apply the accessory_keywords exclusion), or
- **Keep the whole bucket** and rename it appropriately in the catalogue

Default plan: filter to likely-phones if their count is sufficient (~100+ after cleaning) for the sampling step.

## 8. Brand Extraction Preview

Brand is not a separate column in the raw data; it's embedded in the title (usually as the first word or phrase). For the catalogue, we want a clean `brand` field so tasks can include constraints like `brand=Sony`. 

Check feasibility: how often does the first word of the title look like a brand?

In [ ]:
# Extract first 1-2 words as a brand candidate
df_targets['brand_candidate'] = df_targets['title'].fillna('').str.split().str[0]

print('Top 20 brand candidates per bucket:')
for bucket in sorted(df_targets['bucket'].unique()):
    print(f'\n--- {bucket} ---')
    top_brands = df_targets[df_targets['bucket'] == bucket]['brand_candidate'].value_counts().head(20)
    print(top_brands.to_string())

**Observations to capture:**
- Are recognisable brand names appearing at the top (Sony, Apple, Samsung, etc.)?
- Or is there a lot of noise (generic words like "The", "New", random characters)?
- For Watches especially: brand names may often be 2-word phrases (e.g., "Casio Watch", "Citizen Eco-Drive")

The pre-processing pipeline will use this finding to decide whether first-word extraction is good enough, or whether we need a curated brand list per category.

## 9. Summary Findings

Final cell: aggregate the key findings into one block. These drive the design of the pre-processing pipeline and inform the EDA report.

**Manually fill in observations after running the cells above.** Suggested template:

1. **Coverage:** XX,XXX products across our 5 buckets after raw filtering — comfortable for sampling ~500.
2. **Missing values:** Use zero-value detection rather than NaN-only; price/stars/reviews of 0 are effectively missing.
3. **Duplicates:** N ASINs duplicates, M title duplicates — drop both.
4. **Price distributions:** Per-bucket medians and quantiles — used to set task thresholds.
5. **Rating distributions:** A 4+ star constraint filters XX% of products — adjust if too loose/tight.
6. **Review distributions:** 1000+ reviews is a meaningful constraint at the ~XX percentile.
7. **Phones quirk:** ~XX% of the Phones bucket is accessories — apply keyword filter.
8. **Brand extraction:** First-word extraction looks [feasible/noisy] — proceed with [strategy].
9. **Laptops substitution:** Acknowledged in report — use Laptop Accessories instead, adjust task constraints accordingly.

**Next step:** Encode these findings into the pre-processing pipeline (`src/preprocessing/clean.py`, `sample.py`, `pipeline.py`). This produces `data/processed/catalogue.parquet`, the curated artefact on which the task suite is later generated.